# Notebook 18 – Model Comparison


In [1]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("data.csv", encoding="latin1")
df = df.dropna(subset=["Description"]).copy()
df = df[df["UnitPrice"] > 0]
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["IsCancelled"] = df["InvoiceNo"].astype(str).str.startswith("C").astype(int)
df["AbsQuantity"] = df["Quantity"].abs()
df["Month"] = df["InvoiceDate"].dt.month
df["IsInternational"] = (df["Country"] != "United Kingdom").astype(int)

feature_cols = ["AbsQuantity", "UnitPrice", "Month", "IsInternational"]
target_col = "IsCancelled"
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancelled,AbsQuantity,Month,IsInternational
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,0,6,12,0
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,6,12,0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,0,8,12,0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,6,12,0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,6,12,0


## Setup

**Task:** the same `IsCancelled` binary classification problem used
throughout this sprint. To keep every algorithm here — including SVM,
which trains far more slowly than the others — comparable and fast to
run side by side, a **stratified 20,000-row subsample** is used for this
notebook, preserving the original ~1.7% cancellation rate.

**Split:** 60% train / 20% validation / 20% test, so a genuine
Training Score, Validation Score, and Test Score can each be reported
separately in the comparison table, matching the workflow established in
Notebook 17.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sample_df, _ = train_test_split(df, train_size=20000, random_state=42, stratify=df[target_col])

X = sample_df[feature_cols]
y = sample_df[target_col]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {len(X_train)} rows ({y_train.mean():.4f} cancellation rate)")
print(f"Validation: {len(X_val)} rows ({y_val.mean():.4f} cancellation rate)")
print(f"Test: {len(X_test)} rows ({y_test.mean():.4f} cancellation rate)")

Train: 12000 rows (0.0172 cancellation rate)
Validation: 4000 rows (0.0173 cancellation rate)
Test: 4000 rows (0.0173 cancellation rate)


## Training Five Candidate Algorithms

Five algorithms spanning very different modeling approaches (linear,
probabilistic, tree-based, ensemble, margin-based) are trained on the
identical training data, so the comparison table below is a fair,
apples-to-apples evaluation of the *algorithms* themselves.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

models = {
    "Logistic Regression": (LogisticRegression(max_iter=1000, class_weight="balanced"), True),
    "Naive Bayes (Gaussian)": (GaussianNB(), False),
    "Decision Tree": (DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42), False),
    "Random Forest": (RandomForestClassifier(n_estimators=150, max_depth=8, class_weight="balanced",
                                             random_state=42, n_jobs=-1), False),
    "SVM (RBF Kernel)": (SVC(kernel="rbf", C=1.0, class_weight="balanced", probability=False, random_state=42), True),
}
# The boolean flags whether that algorithm needs SCALED features (distance/gradient-based models do; tree-based models don't)

trained = {}
training_times = {}

for name, (model, needs_scaling) in models.items():
    Xtr = X_train_scaled if needs_scaling else X_train
    start = time.time()
    model.fit(Xtr, y_train)
    training_times[name] = time.time() - start
    trained[name] = model
    print(f"{name:25s} trained in {training_times[name]:.3f} seconds")

Logistic Regression       trained in 0.504 seconds
Naive Bayes (Gaussian)    trained in 0.052 seconds
Decision Tree             trained in 0.087 seconds
Random Forest             trained in 1.450 seconds
SVM (RBF Kernel)          trained in 23.036 seconds


## Model Complexity

Before scoring, each model's **complexity** is captured in simple,
comparable terms — the number of internal parameters or structural size
that drives how flexible (and how opaque) each model is, tying back
directly to Notebook 15's discussion of complexity and the bias-variance
tradeoff.

In [4]:
def describe_complexity(name, model):
    if name == "Logistic Regression":
        return f"{model.coef_.size + 1} parameters (linear)"
    if name == "Naive Bayes (Gaussian)":
        return f"{model.theta_.size * 2} parameters (per-class mean/variance)"
    if name == "Decision Tree":
        return f"{model.get_n_leaves()} leaves, depth {model.get_depth()}"
    if name == "Random Forest":
        avg_leaves = np.mean([t.get_n_leaves() for t in model.estimators_])
        return f"{len(model.estimators_)} trees, avg {avg_leaves:.0f} leaves each"
    if name == "SVM (RBF Kernel)":
        return f"{len(model.support_vectors_)} support vectors"
    return "n/a"

complexity_notes = {name: describe_complexity(name, model) for name, model in trained.items()}
for name, note in complexity_notes.items():
    print(f"{name:25s} {note}")

Logistic Regression       5 parameters (linear)
Naive Bayes (Gaussian)    16 parameters (per-class mean/variance)
Decision Tree             38 leaves, depth 6
Random Forest             150 trees, avg 79 leaves each
SVM (RBF Kernel)          10065 support vectors


## Building the Comparison Table

The table below reports every score requested: **Training / Validation /
Test Score** (accuracy), **Accuracy** again as its own explicit column,
**Precision / Recall / F1** (all on the Test set, on the "Cancelled"
class), **MAE / RMSE** (marked N/A — these are regression-only metrics
from Notebook 13, and every model here is a classifier), **Training
Time**, and **Model Complexity**.

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

rows = []
for name, (model, needs_scaling) in models.items():
    Xtr = X_train_scaled if needs_scaling else X_train
    Xva = X_val_scaled if needs_scaling else X_val
    Xte = X_test_scaled if needs_scaling else X_test

    train_acc = accuracy_score(y_train, model.predict(Xtr))
    val_acc = accuracy_score(y_val, model.predict(Xva))
    test_pred = model.predict(Xte)
    test_acc = accuracy_score(y_test, test_pred)

    rows.append({
        "Algorithm": name,
        "Training Score": round(train_acc, 4),
        "Validation Score": round(val_acc, 4),
        "Test Score": round(test_acc, 4),
        "Accuracy": round(test_acc, 4),
        "Precision": round(precision_score(y_test, test_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, test_pred, zero_division=0), 4),
        "F1": round(f1_score(y_test, test_pred, zero_division=0), 4),
        "MAE / RMSE": "N/A (classification task)",
        "Training Time (s)": round(training_times[name], 3),
        "Model Complexity": complexity_notes[name],
    })

comparison_table = pd.DataFrame(rows).set_index("Algorithm")
comparison_table

,Training Score,Validation Score,Test Score,Accuracy,Precision,Recall,F1,MAE / RMSE,Training Time (s),Model Complexity
Algorithm,,,,,,,,,,
Logistic Regression,0.7622,0.7540,0.7602,0.7602,0.0256,0.3478,0.0477,N/A (classification task),0.504,5 parameters (linear)
Naive Bayes (Gaussian),0.9786,0.9765,0.9762,0.9762,0.0357,0.0145,0.0206,N/A (classification task),0.052,16 parameters (per-class mean/variance)
Decision Tree,0.7322,0.7262,0.7275,0.7275,0.0278,0.4348,0.0522,N/A (classification task),0.087,"38 leaves, depth 6"
Random Forest,0.8250,0.8187,0.8185,0.8185,0.0300,0.3043,0.0547,N/A (classification task),1.450,"150 trees, avg 79 leaves each"
SVM (RBF Kernel),0.8253,0.8257,0.8165,0.8165,0.0243,0.2464,0.0443,N/A (classification task),23.036,10065 support vectors


## Explaining the Selection — Why Not Just Pick the Highest Accuracy?

Reading the table by **Accuracy alone** would point toward whichever
model predicts "Not Cancelled" most often, since cancellations are rare
(~1.7%) — exactly the trap Notebook 14 warned about. A model can score
very high accuracy while barely detecting any real cancellations at all.
Selecting a final model requires looking across *several* columns at
once, not defaulting to the single highest number in one of them.

In [6]:
print("Sorted by Accuracy (misleading on its own):")
print(comparison_table.sort_values("Accuracy", ascending=False)[["Accuracy", "Recall", "F1"]])

print("\nSorted by F1 (balances Precision and Recall -- a fairer summary for this imbalanced task):")
print(comparison_table.sort_values("F1", ascending=False)[["Accuracy", "Recall", "F1"]])

Sorted by Accuracy (misleading on its own):
                        Accuracy  Recall      F1
Algorithm                                       
Naive Bayes (Gaussian)    0.9762  0.0145  0.0206
Random Forest             0.8185  0.3043  0.0547
SVM (RBF Kernel)          0.8165  0.2464  0.0443
Logistic Regression       0.7602  0.3478  0.0477
Decision Tree             0.7275  0.4348  0.0522

Sorted by F1 (balances Precision and Recall -- a fairer summary for this imbalanced task):
                        Accuracy  Recall      F1
Algorithm                                       
Random Forest             0.8185  0.3043  0.0547
Decision Tree             0.7275  0.4348  0.0522
Logistic Regression       0.7602  0.3478  0.0477
SVM (RBF Kernel)          0.8165  0.2464  0.0443
Naive Bayes (Gaussian)    0.9762  0.0145  0.0206
